In [4]:
# Diabetic Patient Follow-up Analysis
#This notebook breaks down the logic for identifying diabetic patients who are overdue for follow-up.
import pandas as pd
import re
from datetime import datetime

print("Loading hospital data...")
# Load data
patients = pd.read_csv('export/patients.csv')
visits = pd.read_csv('export/visits.csv')
labs = pd.read_csv('export/lab_results.csv')
rx = pd.read_csv('export/prescriptions.csv')

Loading hospital data...


In [5]:
# 1. Clean Data & Fix Formats
#Fixing phone number formatting and safely parsing dates to prevent UK/US format swaps.
# Phone numbers parsed as floats (e.g., 9.000085e+09) due to NaNs. Convert to clean strings.
patients['phone_str'] = patients['phone'].fillna(0).astype(int).astype(str)
patients['phone_str'] = patients['phone_str'].replace('0', 'No Phone on File')

# Parse dates safely (handling DD/MM/YYYY vs YYYY-MM-DD variations)
visits['visit_date_dt'] = pd.to_datetime(visits['visit_date'], dayfirst=True, errors='coerce')
visits['next_appt_dt'] = pd.to_datetime(visits['next_appointment'], dayfirst=True, errors='coerce')

# Determine reference "today" date based on the latest record (assumes data was exported ~Oct 21, 2026)
today = pd.to_datetime('2026-10-21')

In [6]:
# 2. Identify Diabetic Patients (Multi-modal approach)
#Since there isn't a single "is_diabetic" flag, we look across Notes, Labs, and Prescriptions.
# A. From Visit Notes
note_pattern = re.compile(r'\b(diabet\w*|t2dm|t1dm|dm)\b', re.IGNORECASE)
visits['is_diabetic_note'] = visits['notes'].apply(lambda x: bool(note_pattern.search(str(x))))
diabetic_from_notes = visits[visits['is_diabetic_note']]['mrn'].unique()

# B. From Lab Results (HbA1c >= 6.5, Fasting Glucose >= 126, PP Glucose >= 200)
labs['numeric_value'] = pd.to_numeric(labs['value'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
is_high_hba1c = labs['test_name'].isin(['A1C', 'HbA1c', 'Glycated Hb']) & (labs['numeric_value'] >= 6.5)
is_high_fasting = labs['test_name'].isin(['Glucose (F)', 'FBS', 'Fasting Glucose']) & (labs['numeric_value'] >= 126)
is_high_pp = labs['test_name'].isin(['Glucose (PP)', 'PPBS', 'Post Prandial Glucose']) & (labs['numeric_value'] >= 200)
diabetic_from_labs = labs[is_high_hba1c | is_high_fasting | is_high_pp]['mrn'].unique()

# C. From Prescriptions
diabetes_drugs = ['vildagliptin', 'metformin', 'glimepiride', 'mixtard', 'dapagliflozin', 'sitagliptin', 'insulin', 'gliclazide', 'pioglitazone']
drug_pattern = re.compile(r'\b(' + '|'.join(diabetes_drugs) + r')\b', re.IGNORECASE)
rx['is_diabetic_drug'] = rx['drug'].apply(lambda x: bool(drug_pattern.search(str(x))))
diabetic_from_rx = rx[rx['is_diabetic_drug']]['mrn'].unique()

# Combine all unique diabetic patients
all_diabetic_mrns = set(diabetic_from_notes) | set(diabetic_from_labs) | set(diabetic_from_rx)
print(f"Identified {len(all_diabetic_mrns)} diabetic patients.")

Identified 115 diabetic patients.


In [10]:
# Filter visits for diabetics, drop rows with unparseable dates, and find their absolute latest visit
diabetic_visits = visits[visits['mrn'].isin(all_diabetic_mrns)].dropna(subset=['visit_date_dt'])
latest_visits = diabetic_visits.loc[diabetic_visits.groupby('mrn')['visit_date_dt'].idxmax()].copy()

# Check who has future appointments scheduled anywhere in the system to avoid calling them
future_appts = visits[visits['next_appt_dt'] > today]['mrn'].unique()

latest_visits['days_since_last'] = (today - latest_visits['visit_date_dt']).dt.days
latest_visits['has_future_appt'] = latest_visits['mrn'].isin(future_appts)
latest_visits['missed_past_appt'] = (latest_visits['next_appt_dt'] < today) & latest_visits['next_appt_dt'].notna()

# Logic: Overdue IF (Missed scheduled appt OR (> 180 days since visit with no appt scheduled)) AND no future appt booked.
overdue_mask = (~latest_visits['has_future_appt']) & (
    latest_visits['missed_past_appt'] | 
    ((latest_visits['next_appt_dt'].isna()) & (latest_visits['days_since_last'] > 180))
)
overdue_mrns = latest_visits[overdue_mask]['mrn']

In [11]:
final_data = latest_visits[latest_visits['mrn'].isin(overdue_mrns)][['mrn', 'visit_date', 'next_appointment', 'days_since_last', 'missed_past_appt']]
final_data = pd.merge(final_data, patients[['mrn', 'name', 'phone_str']], on='mrn', how='left')

def generate_reason(row):
    if row['missed_past_appt']:
        return f"Missed scheduled appointment on {row['next_appointment']}"
    return f"No visit in {row['days_since_last']} days (Last: {row['visit_date']})"

final_data['Call Reason'] = final_data.apply(generate_reason, axis=1)

# Clean up columns for human readability
front_desk_list = final_data[['name', 'phone_str', 'Call Reason', 'visit_date']]
front_desk_list.columns = ['Patient Name', 'Phone Number', 'Context for Call', 'Last Visit Date']
front_desk_list = front_desk_list.sort_values('Context for Call')

output_file = 'overdue_diabetics_call_list.csv'
front_desk_list.to_csv(output_file, index=False)
print(f"Success! Generated {output_file} with {len(front_desk_list)} overdue patients.")
front_desk_list.head()

Success! Generated overdue_diabetics_call_list.csv with 73 overdue patients.


,Patient Name,Phone Number,Context for Call,Last Visit Date
4,Gracy F,9000006353,Missed scheduled appointment on 2023-11-07,2023-10-10
15,Revathi Pandian,9000043929,Missed scheduled appointment on 2024-09-03,2024-06-04
35,Subramanian Angadi,9000043812,Missed scheduled appointment on 2025-02-02,2024-12-03
40,Geetha Kattimani,9000001489,Missed scheduled appointment on 2025-06-02,2025-03-05
1,N. Anitha,9000076203,Missed scheduled appointment on 2025-10-01,2025-07-03
